In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
# Load dataset
df = pd.read_csv("Local Special Rice.csv")

print(df.head())
print(df.columns.tolist())

        Date  Special_PHP_kg_L  Brent_Oil_USD  Farmgate_LCU_tonne  \
0  8/26/2023             57.80          82.50               17740   
1   9/2/2023             58.51          82.59               18190   
2   9/9/2023             58.25          82.59               18190   
3  9/16/2023             58.46          82.59               18190   
4  9/23/2023             58.24          82.59               18190   

   Inflation_Rate  Stocks_MT  USD to PHP  Temp_C  Rainfall_mm     VoP_MT  
0             8.7   651930.0     56.1599   26.65         6.22  292142.87  
1             8.6   541210.0     56.7869   26.62        14.99  292142.87  
2             8.6   541210.0     56.7869   27.21         5.78  292142.87  
3             8.6   541210.0     56.7869   26.97         5.44  292142.87  
4             8.6   541210.0     56.7869   27.81         8.84  292142.87  
['Date', 'Special_PHP_kg_L', 'Brent_Oil_USD', 'Farmgate_LCU_tonne', 'Inflation_Rate', 'Stocks_MT', 'USD to PHP', 'Temp_C', 'Rainfall_mm

In [3]:
# Remove the Date column
df.drop(columns=['Date'], inplace=True)

# Dropping target variable 
X = df.drop('Special_PHP_kg_L', axis=1)
y = df['Special_PHP_kg_L']

# Make sure X and y are sorted by date already
n = len(X)

# Index cutoffs
train_end = int(n * 0.80)   # 80% mark
val_end   = int(n * 0.90)   # 80% + 10% mark

# Chronological splits
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val     = X[train_end:val_end], y[train_end:val_end]
X_test, y_test   = X[val_end:], y[val_end:]

print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

Train size: 119 | Val size: 15 | Test size: 15


In [4]:
def walk_forward_validation(X, y, initial_train_size, test_size=1):
  y_true_all = []
  y_pred_all = []
  results = []

  for i in range(initial_train_size, len(X) - test_size + 1):
    X_tr, y_tr = X[:i], y[:i]
    X_te, y_te = X[i : i + test_size], y[i : i + test_size]

    model = xgb.XGBRegressor(objective="reg:squarederror")
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_te)

    # Store individual predictions
    y_true_all.extend(y_te)
    y_pred_all.extend(y_pred)

    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae = mean_absolute_error(y_te, y_pred)
    results.append({"step": i, "RMSE": rmse, "MAE": mae})

  df_res = pd.DataFrame(results)

  # Compute overall global R2 across all walk-forward steps
  overall_r2 = r2_score(y_true_all, y_pred_all)

  print(f"Overall Walk-Forward R2 : {overall_r2:.4f}")
  print(f"Average RMSE            : {df_res['RMSE'].mean():.4f}")
  print(f"Average MAE             : {df_res['MAE'].mean():.4f}")

  return df_res

In [5]:
# Run walk-forward validation starting at 80% of the dataset
df_results = walk_forward_validation(
    X, y, initial_train_size=int(len(X) * 0.80)
)

Overall Walk-Forward R2 : 0.7845
Average RMSE            : 0.3998
Average MAE             : 0.3998


## Improved evaluation and charts
Added for review: proper overall scores (fixes RMSE = MAE), comparison with a naive baseline ("price = last week"), and charts for Chapter 4.

In [6]:
%pip install matplotlib

Note: you may need to restart the kernel to use updated packages.


### Cell 1: Overall scores + naive baseline comparison

In [7]:
import matplotlib.pyplot as plt

# Walk-forward that keeps every prediction, so we can score and chart them
def walk_forward_predictions(X, y, start):
    preds = []
    for i in range(start, len(X)):
        model = xgb.XGBRegressor(objective="reg:squarederror")
        model.fit(X[:i], y[:i])
        preds.append(model.predict(X[i:i+1])[0])
    return np.array(preds), model

start = int(len(X) * 0.80)
all_dates = pd.to_datetime(pd.read_csv("Local Special Rice.csv")["Date"])
dates = all_dates.iloc[start:]
actual = y.iloc[start:].values
xgb_pred, last_model = walk_forward_predictions(X, y, start)
naive_pred = X["Special_PHP_kg_L_lag1"].iloc[start:].values

def scores(a, p):
    return {"MAE": mean_absolute_error(a, p),
            "RMSE": np.sqrt(mean_squared_error(a, p)),
            "MAPE (%)": np.mean(np.abs((a - p) / a)) * 100,
            "R2": r2_score(a, p)}

results = pd.DataFrame({"XGBoost": scores(actual, xgb_pred),
                        "Naive (same as last week)": scores(actual, naive_pred)}).T.round(4)
results

KeyError: 'Special_PHP_kg_L_lag1'

### Cell 2: Actual vs predicted

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(dates, actual, marker="o", linewidth=2, label="Actual price")
plt.plot(dates, xgb_pred, marker="s", linestyle="--", label="XGBoost")
plt.plot(dates, naive_pred, linestyle=":", label="Naive (same as last week)")
plt.title("Local Special Rice: Actual vs Predicted (Walk-Forward Test)")
plt.xlabel("Week"); plt.ylabel("PHP per kg")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Cell 3: Full price history with test period

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(all_dates, y, label="Actual price")
plt.plot(dates, xgb_pred, "--", label="XGBoost forecast")
plt.axvspan(dates.iloc[0], dates.iloc[-1], alpha=0.15, color="orange", label="Test period")
plt.title("Local Special Rice Price, Sep 2023 – Jun 2026")
plt.ylabel("PHP per kg"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Cell 4: Weekly errors and feature importance

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

err = actual - xgb_pred
ax[0].bar(dates, err, width=5, color=["tab:red" if e < 0 else "tab:green" for e in err])
ax[0].axhline(0, color="black", linewidth=0.8)
ax[0].set_title("Prediction Error per Week (Actual − Predicted)")
ax[0].set_ylabel("PHP per kg")

imp = pd.Series(last_model.feature_importances_, index=X.columns).sort_values().tail(10)
imp.plot.barh(ax=ax[1], title="Top 10 Most Important Features")

plt.tight_layout(); plt.show()